# LightGBM Prediction Interval Quantile Loss

### Loading Libraries

In [1]:
%cd ../..

/Users/joaquinromero/Desktop


#### Installing MLForecast (if not installed)

In [2]:
%%capture
!pip install -q mlforecast

In [6]:
# Numerical Computing
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt

# OS
import os

In [5]:
# Custom Libraries
from src.utils.plotting_utils import plot_grid

from src.utils.ts_utils import average_length, level_to_quantiles

### Loading M4 Data

In [ ]:
from src.utils.general import download_m4_data

if not os.path.exists('data/m4/M4-Hourly.csv'):
    download_m4_data()
Y_train_df = pd.read_csv('data/m4/M4-Hourly.csv')
Y_test_df = pd.read_csv('data/m4/M4-Hourly-test.csv')#.rename(columns={'y': 'y_test'})

In [ ]:
# Taking first n_series time series from the dataset
n_series = 8
uids = Y_train_df['unique_id'].unique()[:n_series]
Y_train_df = Y_train_df.query('unique_id in @uids')
Y_test_df = Y_test_df.query('unique_id in @uids')

#### Plotting `Sampled TS`

In [ ]:
# Plotting the selected time series
plot_grid(Y_train_df, Y_test_df)

#### Adding a Few Features We Learned to Make this Into a Regression Problem

In [ ]:
from mlforecast import MLForecast

from mlforecast.lag_transforms import RollingMean, ExpandingMean, Offset

In [ ]:
data = pd.concat([Y_train_df, Y_test_df], ignore_index=True)

In [ ]:
fcst = MLForecast(
    models=[],
    freq=24,
    lags=[1,2,3,4, 7], # Defining the Lags we need to create
    # Defining some transformations we need to do to the lags (offsets)
    lag_transforms={
        1: [ExpandingMean(), RollingMean(window_size=7, min_samples=1), RollingMean(window_size=14)],
    },
)
data = fcst.preprocess(data)
data.head()

### LightGBM

In [ ]:
import lightgbm as lgb

from lightgbm import LGBMRegressor

In [ ]:
# Splitting Train and Val after feature engineering
val_dates = Y_test_df.ds.unique()

train = data[~data['ds'].isin(val_dates)]
val = data[data['ds'].isin(val_dates)]
train_keys = train[["unique_id", 'ds']]
val_keys = val[["unique_id", 'ds']]
train = train.drop(['unique_id', 'ds'], axis=1)
val = val.drop(['unique_id', 'ds'], axis=1)
X_train, Y_train = train.drop(['y'], axis=1), train['y']
X_val, Y_val = val.drop(['y'], axis=1), val['y']